In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import MinMaxScaler, MultiLabelBinarizer

Ссылка на датасет: [Games Rating Dataset](https://www.kaggle.com/competitions/games-rating)

In [2]:
df = pd.read_csv("train_data.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15257 entries, 0 to 15256
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ID                  15247 non-null  float64
 1   Name                15257 non-null  object 
 2   Year Published      15256 non-null  float64
 3   Min Players         15257 non-null  int64  
 4   Max Players         15257 non-null  int64  
 5   Play Time           15257 non-null  int64  
 6   Min Age             15257 non-null  int64  
 7   Users Rated         15257 non-null  int64  
 8   Rating Average      15257 non-null  object 
 9   BGG Rank            15257 non-null  int64  
 10  Complexity Average  15257 non-null  object 
 11  Owned Users         15240 non-null  float64
 12  Mechanics           14057 non-null  object 
 13  Domains             7608 non-null   object 
dtypes: float64(3), int64(6), object(5)
memory usage: 1.6+ MB


In [3]:
df.head()

,ID,Name,Year Published,Min Players,Max Players,Play Time,Min Age,Users Rated,Rating Average,BGG Rank,Complexity Average,Owned Users,Mechanics,Domains
0,174430.0,Gloomhaven,2017.0,1,4,120,14,42055,"8,79",1,"3,86",68323.0,"Action Queue, Action Retrieval, Campaign / Bat...","Strategy Games, Thematic Games"
1,224517.0,Brass: Birmingham,2018.0,2,4,120,14,19217,"8,66",3,"3,91",28785.0,"Hand Management, Income, Loans, Market, Networ...",Strategy Games
2,167791.0,Terraforming Mars,2016.0,1,5,120,12,64864,"8,43",4,"3,24",87099.0,"Card Drafting, Drafting, End Game Bonuses, Han...",Strategy Games
3,233078.0,Twilight Imperium: Fourth Edition,2017.0,3,6,480,14,13468,"8,70",5,"4,22",16831.0,"Action Drafting, Area Majority / Influence, Ar...","Strategy Games, Thematic Games"
4,291457.0,Gloomhaven: Jaws of the Lion,2020.0,1,4,120,14,8392,"8,87",6,"3,55",21609.0,"Action Queue, Campaign / Battle Card Driven, C...","Strategy Games, Thematic Games"


In [4]:
df["Rating Average"] = df["Rating Average"].str.replace(",", ".").astype(float)
df["Complexity Average"] = df["Complexity Average"].str.replace(",", ".").astype(float)

In [5]:
(df.isnull().mean() * 100).round(2)
# Проценты пропусков

ID                     0.07
Name                   0.00
Year Published         0.01
Min Players            0.00
Max Players            0.00
Play Time              0.00
Min Age                0.00
Users Rated            0.00
Rating Average         0.00
BGG Rank               0.00
Complexity Average     0.00
Owned Users            0.11
Mechanics              7.87
Domains               50.13
dtype: float64

In [6]:
df.isnull().sum()

ID                      10
Name                     0
Year Published           1
Min Players              0
Max Players              0
Play Time                0
Min Age                  0
Users Rated              0
Rating Average           0
BGG Rank                 0
Complexity Average       0
Owned Users             17
Mechanics             1200
Domains               7649
dtype: int64

In [7]:
df[["Domains"]]

,Domains
0,"Strategy Games, Thematic Games"
1,Strategy Games
2,Strategy Games
3,"Strategy Games, Thematic Games"
4,"Strategy Games, Thematic Games"
...,...
15252,Children's Games
15253,Children's Games
15254,Children's Games
15255,Children's Games


In [8]:
df["Domains_splited"] = df["Domains"].str.split(", ")
df["Domains_splited"] =  df["Domains_splited"].apply(
    lambda x: x if isinstance(x, list) else ["MISSING"]
)

mlb_Domains = MultiLabelBinarizer()
domains_encoded = mlb_Domains.fit_transform(df["Domains_splited"])

domains_df = pd.DataFrame(
    domains_encoded,
    columns=[f"DOMAIN_{cls}" for cls in mlb_Domains.classes_]
)

df = pd.concat([df, domains_df], axis=1)
df = df.drop(columns=["Domains", "Domains_splited"])

In [9]:
df.head()

,ID,Name,Year Published,Min Players,Max Players,Play Time,Min Age,Users Rated,Rating Average,BGG Rank,...,Mechanics,DOMAIN_Abstract Games,DOMAIN_Children's Games,DOMAIN_Customizable Games,DOMAIN_Family Games,DOMAIN_MISSING,DOMAIN_Party Games,DOMAIN_Strategy Games,DOMAIN_Thematic Games,DOMAIN_Wargames
0,174430.0,Gloomhaven,2017.0,1,4,120,14,42055,8.79,1,...,"Action Queue, Action Retrieval, Campaign / Bat...",0,0,0,0,0,0,1,1,0
1,224517.0,Brass: Birmingham,2018.0,2,4,120,14,19217,8.66,3,...,"Hand Management, Income, Loans, Market, Networ...",0,0,0,0,0,0,1,0,0
2,167791.0,Terraforming Mars,2016.0,1,5,120,12,64864,8.43,4,...,"Card Drafting, Drafting, End Game Bonuses, Han...",0,0,0,0,0,0,1,0,0
3,233078.0,Twilight Imperium: Fourth Edition,2017.0,3,6,480,14,13468,8.70,5,...,"Action Drafting, Area Majority / Influence, Ar...",0,0,0,0,0,0,1,1,0
4,291457.0,Gloomhaven: Jaws of the Lion,2020.0,1,4,120,14,8392,8.87,6,...,"Action Queue, Campaign / Battle Card Driven, C...",0,0,0,0,0,0,1,1,0


In [10]:
df[["Mechanics"]]

,Mechanics
0,"Action Queue, Action Retrieval, Campaign / Bat..."
1,"Hand Management, Income, Loans, Market, Networ..."
2,"Card Drafting, Drafting, End Game Bonuses, Han..."
3,"Action Drafting, Area Majority / Influence, Ar..."
4,"Action Queue, Campaign / Battle Card Driven, C..."
...,...
15252,Roll / Spin and Move
15253,NaN
15254,Roll / Spin and Move
15255,"Dice Rolling, Grid Movement, Race, Roll / Spin..."


In [11]:
df["Mechanic_split"] = df["Mechanics"].str.split(", ")
df["Mechanic_split"] = df["Mechanic_split"].apply(
    lambda x: x if isinstance(x, list) else ["MISSING"]
)

all_mechanics_series = df["Mechanic_split"].explode()
top10_mechanics = all_mechanics_series.value_counts().head(10).index

df["Mechanics_top10"] = df["Mechanic_split"].apply(
    lambda lst: [m for m in lst if m in top10_mechanics]
)

mlb_Mechanic = MultiLabelBinarizer()
mechanic_encoded = mlb_Mechanic.fit_transform(df["Mechanics_top10"])

mechanics_df = pd.DataFrame(
    mechanic_encoded,
    columns=[f"MECHANIC_{cls}" for cls in mlb_Mechanic.classes_]
)

df = pd.concat([df, mechanics_df], axis=1)
df = df.drop(columns=["Mechanics", "Mechanics_top10", "Mechanic_split"])

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15257 entries, 0 to 15256
Data columns (total 31 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   ID                               15247 non-null  float64
 1   Name                             15257 non-null  object 
 2   Year Published                   15256 non-null  float64
 3   Min Players                      15257 non-null  int64  
 4   Max Players                      15257 non-null  int64  
 5   Play Time                        15257 non-null  int64  
 6   Min Age                          15257 non-null  int64  
 7   Users Rated                      15257 non-null  int64  
 8   Rating Average                   15257 non-null  float64
 9   BGG Rank                         15257 non-null  int64  
 10  Complexity Average               15257 non-null  float64
 11  Owned Users                      15240 non-null  float64
 12  DOMAIN_Abstract Ga

In [13]:
df = df.drop(columns=["ID", "Name"])

In [14]:
df.isna().sum()

Year Published                      1
Min Players                         0
Max Players                         0
Play Time                           0
Min Age                             0
Users Rated                         0
Rating Average                      0
BGG Rank                            0
Complexity Average                  0
Owned Users                        17
DOMAIN_Abstract Games               0
DOMAIN_Children's Games             0
DOMAIN_Customizable Games           0
DOMAIN_Family Games                 0
DOMAIN_MISSING                      0
DOMAIN_Party Games                  0
DOMAIN_Strategy Games               0
DOMAIN_Thematic Games               0
DOMAIN_Wargames                     0
MECHANIC_Card Drafting              0
MECHANIC_Dice Rolling               0
MECHANIC_Hand Management            0
MECHANIC_Hexagon Grid               0
MECHANIC_MISSING                    0
MECHANIC_Modular Board              0
MECHANIC_Set Collection             0
MECHANIC_Sim

In [15]:
df.describe()

,Year Published,Min Players,Max Players,Play Time,Min Age,Users Rated,Rating Average,BGG Rank,Complexity Average,Owned Users,...,MECHANIC_Card Drafting,MECHANIC_Dice Rolling,MECHANIC_Hand Management,MECHANIC_Hexagon Grid,MECHANIC_MISSING,MECHANIC_Modular Board,MECHANIC_Set Collection,MECHANIC_Simulation,MECHANIC_Tile Placement,MECHANIC_Variable Player Powers
count,15256.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15240.000000,...,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000
mean,1984.144140,2.021236,5.642918,92.801337,9.581766,819.530773,6.399775,10186.327784,1.991064,1372.209843,...,0.084158,0.280330,0.204365,0.110179,0.078652,0.080946,0.132792,0.095759,0.084551,0.125188
std,211.725679,0.696323,12.551944,607.611378,3.671560,3286.396397,0.936488,5863.744570,0.848816,4681.296849,...,0.277634,0.449175,0.403250,0.313123,0.269204,0.272762,0.339360,0.294271,0.278222,0.330943
min,-3500.000000,0.000000,0.000000,0.000000,0.000000,30.000000,1.050000,1.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2001.000000,2.000000,4.000000,30.000000,8.000000,55.000000,5.820000,5103.000000,1.330000,146.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2011.000000,2.000000,4.000000,45.000000,10.000000,119.000000,6.430000,10171.000000,2.000000,308.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,2016.000000,2.000000,6.000000,90.000000,12.000000,379.000000,7.020000,15280.000000,2.540000,856.250000,...,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,2022.000000,10.000000,999.000000,60000.000000,25.000000,102214.000000,9.580000,20344.000000,5.000000,155312.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


Еще очень много невалидных и неправдоподобных значений в таких столбцах как
- Min Players (нули)
- Max Players (нули || < Min Players)
- Play Time (60 тыс часов ?!)

In [16]:
df["Min Players"] = df["Min Players"].replace(0, 1)
df["Max Players"] = np.where(df["Max Players"] == 0, df["Min Players"], df["Max Players"])

df["Play Time"] = df["Play Time"].replace(0, df["Play Time"].median())

df["Min Age"] = df["Min Age"].replace(0, df["Min Age"].median())

In [17]:
df.describe()

,Year Published,Min Players,Max Players,Play Time,Min Age,Users Rated,Rating Average,BGG Rank,Complexity Average,Owned Users,...,MECHANIC_Card Drafting,MECHANIC_Dice Rolling,MECHANIC_Hand Management,MECHANIC_Hexagon Grid,MECHANIC_MISSING,MECHANIC_Modular Board,MECHANIC_Set Collection,MECHANIC_Simulation,MECHANIC_Tile Placement,MECHANIC_Variable Player Powers
count,15256.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15240.000000,...,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000
mean,1984.144140,2.023399,5.657796,94.004719,10.221472,819.530773,6.399775,10186.327784,1.991064,1372.209843,...,0.084158,0.280330,0.204365,0.110179,0.078652,0.080946,0.132792,0.095759,0.084551,0.125188
std,211.725679,0.691578,12.546746,607.470928,2.684911,3286.396397,0.936488,5863.744570,0.848816,4681.296849,...,0.277634,0.449175,0.403250,0.313123,0.269204,0.272762,0.339360,0.294271,0.278222,0.330943
min,-3500.000000,1.000000,1.000000,1.000000,1.000000,30.000000,1.050000,1.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2001.000000,2.000000,4.000000,30.000000,8.000000,55.000000,5.820000,5103.000000,1.330000,146.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2011.000000,2.000000,4.000000,45.000000,10.000000,119.000000,6.430000,10171.000000,2.000000,308.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,2016.000000,2.000000,6.000000,90.000000,12.000000,379.000000,7.020000,15280.000000,2.540000,856.250000,...,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,2022.000000,10.000000,999.000000,60000.000000,25.000000,102214.000000,9.580000,20344.000000,5.000000,155312.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [18]:
df[df["Year Published"] < 0]    # Это реальные игры, их нельзя выкидывать

,Year Published,Min Players,Max Players,Play Time,Min Age,Users Rated,Rating Average,BGG Rank,Complexity Average,Owned Users,...,MECHANIC_Card Drafting,MECHANIC_Dice Rolling,MECHANIC_Hand Management,MECHANIC_Hexagon Grid,MECHANIC_MISSING,MECHANIC_Modular Board,MECHANIC_Set Collection,MECHANIC_Simulation,MECHANIC_Tile Placement,MECHANIC_Variable Player Powers
123,-2200.0,2,2,180,8,14843,7.64,173,4.00,20398.0,...,0,0,0,0,0,0,0,0,0,0
6088,-3500.0,2,2,30,6,664,5.82,8176,1.48,1343.0,...,0,1,0,0,0,0,0,0,0,0
11343,-100.0,2,2,20,5,51,6.01,15136,2.17,93.0,...,0,0,0,0,0,0,0,0,0,0
14740,-1400.0,2,2,5,5,60,4.31,19650,1.25,60.0,...,0,0,0,0,0,0,0,0,0,0
15007,-1400.0,2,2,20,6,1310,5.36,20004,1.84,1642.0,...,0,0,0,0,0,0,0,0,0,0
15255,-200.0,2,6,30,3,3783,2.86,20343,1.02,4400.0,...,0,1,0,0,0,0,0,0,0,0
15256,-1300.0,2,2,1,4,3275,2.68,20344,1.16,1374.0,...,0,0,0,0,0,0,0,0,0,0


In [19]:
df[df["Year Published"] == 0]    # А вот это невалидные значения...

,Year Published,Min Players,Max Players,Play Time,Min Age,Users Rated,Rating Average,BGG Rank,Complexity Average,Owned Users,...,MECHANIC_Card Drafting,MECHANIC_Dice Rolling,MECHANIC_Hand Management,MECHANIC_Hexagon Grid,MECHANIC_MISSING,MECHANIC_Modular Board,MECHANIC_Set Collection,MECHANIC_Simulation,MECHANIC_Tile Placement,MECHANIC_Variable Player Powers
773,0.0,3,99,20,10,1589,7.45,1044,1.11,803.0,...,0,0,0,0,0,0,0,0,0,0
1121,0.0,2,4,60,6,1600,7.04,1489,1.48,1787.0,...,0,0,0,0,0,0,0,0,0,0
2107,0.0,1,1,45,10,804,6.90,2841,2.47,1194.0,...,0,0,0,0,1,0,0,0,0,0
2222,0.0,4,4,120,8,306,8.36,2995,3.33,334.0,...,0,0,1,0,0,0,1,0,0,0
2662,0.0,1,1,45,10,580,6.68,3589,1.70,2670.0,...,0,0,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15095,0.0,2,12,10,8,480,5.01,20128,1.25,961.0,...,0,1,0,0,0,0,0,0,0,0
15129,0.0,2,6,160,15,75,1.55,20176,0.00,3.0,...,0,1,0,0,0,0,0,0,0,0
15154,0.0,2,5,90,10,145,3.02,20206,2.83,70.0,...,0,1,0,0,0,1,0,1,0,1
15247,0.0,2,6,20,4,1445,3.61,20334,1.05,1223.0,...,0,0,0,0,0,0,1,0,0,0


In [20]:
yearMedian = df["Year Published"].median()

df["Year Published"] = df["Year Published"].replace(0, yearMedian)
df["Year Published"] = df["Year Published"].replace(np.nan, yearMedian)

In [21]:
df.describe()

,Year Published,Min Players,Max Players,Play Time,Min Age,Users Rated,Rating Average,BGG Rank,Complexity Average,Owned Users,...,MECHANIC_Card Drafting,MECHANIC_Dice Rolling,MECHANIC_Hand Management,MECHANIC_Hexagon Grid,MECHANIC_MISSING,MECHANIC_Modular Board,MECHANIC_Set Collection,MECHANIC_Simulation,MECHANIC_Tile Placement,MECHANIC_Variable Player Powers
count,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15240.000000,...,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000
mean,2002.994494,2.023399,5.657796,94.004719,10.221472,819.530773,6.399775,10186.327784,1.991064,1372.209843,...,0.084158,0.280330,0.204365,0.110179,0.078652,0.080946,0.132792,0.095759,0.084551,0.125188
std,87.034261,0.691578,12.546746,607.470928,2.684911,3286.396397,0.936488,5863.744570,0.848816,4681.296849,...,0.277634,0.449175,0.403250,0.313123,0.269204,0.272762,0.339360,0.294271,0.278222,0.330943
min,-3500.000000,1.000000,1.000000,1.000000,1.000000,30.000000,1.050000,1.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2001.000000,2.000000,4.000000,30.000000,8.000000,55.000000,5.820000,5103.000000,1.330000,146.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2011.000000,2.000000,4.000000,45.000000,10.000000,119.000000,6.430000,10171.000000,2.000000,308.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,2016.000000,2.000000,6.000000,90.000000,12.000000,379.000000,7.020000,15280.000000,2.540000,856.250000,...,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,2022.000000,10.000000,999.000000,60000.000000,25.000000,102214.000000,9.580000,20344.000000,5.000000,155312.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [22]:
df["Age"] = 2025 - df["Year Published"]

In [23]:
# df["Log Age"] = np.log1p(df["Age"])
# пока не буду трогать, проверить без этого

df = df.drop(columns=["Year Published"])

In [24]:
df["Log Age"] = np.log1p(df["Age"])
df = df.drop(columns=["Age"])

In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15257 entries, 0 to 15256
Data columns (total 29 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Min Players                      15257 non-null  int64  
 1   Max Players                      15257 non-null  int64  
 2   Play Time                        15257 non-null  int64  
 3   Min Age                          15257 non-null  int64  
 4   Users Rated                      15257 non-null  int64  
 5   Rating Average                   15257 non-null  float64
 6   BGG Rank                         15257 non-null  int64  
 7   Complexity Average               15257 non-null  float64
 8   Owned Users                      15240 non-null  float64
 9   DOMAIN_Abstract Games            15257 non-null  int64  
 10  DOMAIN_Children's Games          15257 non-null  int64  
 11  DOMAIN_Customizable Games        15257 non-null  int64  
 12  DOMAIN_Family Game

In [26]:
df[df["Owned Users"].isna()]

,Min Players,Max Players,Play Time,Min Age,Users Rated,Rating Average,BGG Rank,Complexity Average,Owned Users,DOMAIN_Abstract Games,...,MECHANIC_Dice Rolling,MECHANIC_Hand Management,MECHANIC_Hexagon Grid,MECHANIC_MISSING,MECHANIC_Modular Board,MECHANIC_Set Collection,MECHANIC_Simulation,MECHANIC_Tile Placement,MECHANIC_Variable Player Powers,Log Age
2100,2,4,45,10,565,7.13,2830,2.00,NaN,0,...,0,1,0,0,0,1,0,0,0,2.302585
2664,2,4,45,10,360,7.20,3592,2.14,NaN,0,...,0,1,0,0,0,1,0,0,0,2.302585
2771,2,4,45,10,336,7.19,3741,2.13,NaN,0,...,0,1,0,0,0,1,0,0,0,2.302585
4331,3,4,360,12,221,6.68,5809,3.00,NaN,0,...,0,0,0,0,0,0,0,0,1,3.295837
6880,2,2,120,12,94,6.72,9204,3.00,NaN,0,...,1,0,1,0,0,0,1,0,0,3.713572
6970,3,12,45,10,216,5.97,9319,1.38,NaN,0,...,0,0,0,0,0,0,0,0,0,3.806662
7555,1,1,60,14,49,7.84,10077,2.83,NaN,0,...,0,0,0,1,0,0,0,0,0,1.609438
8086,2,2,20,10,110,6.26,10778,2.00,NaN,0,...,0,0,0,1,0,0,0,0,0,3.583519
8125,3,8,45,12,137,6.05,10837,2.00,NaN,0,...,0,0,0,1,0,0,0,0,0,3.295837
8762,2,4,30,12,49,7.20,11671,2.00,NaN,0,...,0,0,0,1,0,0,0,0,0,2.639057


In [27]:
df["Owned Users"] = df["Owned Users"].fillna(df["Owned Users"].median())

In [28]:
df.isna().sum()

Min Players                        0
Max Players                        0
Play Time                          0
Min Age                            0
Users Rated                        0
Rating Average                     0
BGG Rank                           0
Complexity Average                 0
Owned Users                        0
DOMAIN_Abstract Games              0
DOMAIN_Children's Games            0
DOMAIN_Customizable Games          0
DOMAIN_Family Games                0
DOMAIN_MISSING                     0
DOMAIN_Party Games                 0
DOMAIN_Strategy Games              0
DOMAIN_Thematic Games              0
DOMAIN_Wargames                    0
MECHANIC_Card Drafting             0
MECHANIC_Dice Rolling              0
MECHANIC_Hand Management           0
MECHANIC_Hexagon Grid              0
MECHANIC_MISSING                   0
MECHANIC_Modular Board             0
MECHANIC_Set Collection            0
MECHANIC_Simulation                0
MECHANIC_Tile Placement            0
M

In [29]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15257 entries, 0 to 15256
Data columns (total 29 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Min Players                      15257 non-null  int64  
 1   Max Players                      15257 non-null  int64  
 2   Play Time                        15257 non-null  int64  
 3   Min Age                          15257 non-null  int64  
 4   Users Rated                      15257 non-null  int64  
 5   Rating Average                   15257 non-null  float64
 6   BGG Rank                         15257 non-null  int64  
 7   Complexity Average               15257 non-null  float64
 8   Owned Users                      15257 non-null  float64
 9   DOMAIN_Abstract Games            15257 non-null  int64  
 10  DOMAIN_Children's Games          15257 non-null  int64  
 11  DOMAIN_Customizable Games        15257 non-null  int64  
 12  DOMAIN_Family Game

In [30]:
df.describe()

,Min Players,Max Players,Play Time,Min Age,Users Rated,Rating Average,BGG Rank,Complexity Average,Owned Users,DOMAIN_Abstract Games,...,MECHANIC_Dice Rolling,MECHANIC_Hand Management,MECHANIC_Hexagon Grid,MECHANIC_MISSING,MECHANIC_Modular Board,MECHANIC_Set Collection,MECHANIC_Simulation,MECHANIC_Tile Placement,MECHANIC_Variable Player Powers,Log Age
count,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,...,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000,15257.000000
mean,2.023399,5.657796,94.004719,10.221472,819.530773,6.399775,10186.327784,1.991064,1371.024055,0.052173,...,0.280330,0.204365,0.110179,0.078652,0.080946,0.132792,0.095759,0.084551,0.125188,2.795870
std,0.691578,12.546746,607.470928,2.684911,3286.396397,0.936488,5863.744570,0.848816,4678.822616,0.222383,...,0.449175,0.403250,0.313123,0.269204,0.272762,0.339360,0.294271,0.278222,0.330943,0.651418
min,1.000000,1.000000,1.000000,1.000000,30.000000,1.050000,1.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.386294
25%,2.000000,4.000000,30.000000,8.000000,55.000000,5.820000,5103.000000,1.330000,146.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.302585
50%,2.000000,4.000000,45.000000,10.000000,119.000000,6.430000,10171.000000,2.000000,308.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.708050
75%,2.000000,6.000000,90.000000,12.000000,379.000000,7.020000,15280.000000,2.540000,855.000000,0.000000,...,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.218876
max,10.000000,999.000000,60000.000000,25.000000,102214.000000,9.580000,20344.000000,5.000000,155312.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,8.617220


In [31]:
df["Max Players"] = df["Max Players"].clip(upper=20)
df["Play Time"] = df["Play Time"].clip(upper=600)

In [32]:
df[df["Play Time"] > 600]

,Min Players,Max Players,Play Time,Min Age,Users Rated,Rating Average,BGG Rank,Complexity Average,Owned Users,DOMAIN_Abstract Games,...,MECHANIC_Dice Rolling,MECHANIC_Hand Management,MECHANIC_Hexagon Grid,MECHANIC_MISSING,MECHANIC_Modular Board,MECHANIC_Set Collection,MECHANIC_Simulation,MECHANIC_Tile Placement,MECHANIC_Variable Player Powers,Log Age


In [33]:
X_train = df.drop(columns="Rating Average")
Y_train = df["Rating Average"]

mmsc = MinMaxScaler()

X_mmsc = mmsc.fit_transform(X_train)
X_mmsc = pd.DataFrame(X_mmsc, columns=X_train.columns)
X_mmsc

,Min Players,Max Players,Play Time,Min Age,Users Rated,BGG Rank,Complexity Average,Owned Users,DOMAIN_Abstract Games,DOMAIN_Children's Games,...,MECHANIC_Dice Rolling,MECHANIC_Hand Management,MECHANIC_Hexagon Grid,MECHANIC_MISSING,MECHANIC_Modular Board,MECHANIC_Set Collection,MECHANIC_Simulation,MECHANIC_Tile Placement,MECHANIC_Variable Player Powers,Log Age
0,0.000000,0.157895,0.198664,0.541667,0.411268,0.000000,0.772,0.439908,0.0,0.0,...,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.112148
1,0.111111,0.157895,0.198664,0.541667,0.187769,0.000098,0.782,0.185337,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.095859
2,0.000000,0.210526,0.198664,0.458333,0.634483,0.000147,0.648,0.560800,0.0,0.0,...,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.126718
3,0.222222,0.263158,0.799666,0.541667,0.131508,0.000197,0.844,0.108369,0.0,0.0,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.112148
4,0.000000,0.157895,0.198664,0.541667,0.081833,0.000246,0.710,0.139133,0.0,0.0,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.056074
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15252,0.111111,0.157895,0.073456,0.125000,0.031561,0.999754,0.210,0.031949,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.376795
15253,0.111111,0.052632,0.048414,0.125000,0.012820,0.999803,0.200,0.002749,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.182792
15254,0.111111,0.157895,0.048414,0.083333,0.038910,0.999902,0.216,0.037267,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.409009
15255,0.111111,0.263158,0.048414,0.083333,0.036728,0.999951,0.204,0.028330,0.0,1.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.874254


In [34]:
model = LinearRegression()
model.fit(X_mmsc, Y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [35]:
test_df = pd.read_csv("test_data.csv")
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5086 entries, 0 to 5085
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ID                  5080 non-null   float64
 1   Name                5086 non-null   object 
 2   Year Published      5086 non-null   float64
 3   Min Players         5086 non-null   int64  
 4   Max Players         5086 non-null   int64  
 5   Play Time           5086 non-null   int64  
 6   Min Age             5086 non-null   int64  
 7   Users Rated         5086 non-null   int64  
 8   BGG Rank            5086 non-null   int64  
 9   Complexity Average  5086 non-null   object 
 10  Owned Users         5080 non-null   float64
 11  Mechanics           4688 non-null   object 
 12  Domains             2576 non-null   object 
dtypes: float64(3), int64(6), object(4)
memory usage: 516.7+ KB


In [36]:
test_df["Complexity Average"] = test_df["Complexity Average"].str.replace(",", ".").astype(float)

In [37]:
test_df.head()

,ID,Name,Year Published,Min Players,Max Players,Play Time,Min Age,Users Rated,BGG Rank,Complexity Average,Owned Users,Mechanics,Domains
0,161936.0,Pandemic Legacy: Season 1,2015.0,2,4,60,13,41643,2,2.84,65294.0,"Action Points, Cooperative Game, Hand Manageme...","Strategy Games, Thematic Games"
1,12333.0,Twilight Struggle,2005.0,2,2,180,13,40814,10,3.59,56219.0,"Action/Event, Advantage Token, Area Majority /...","Strategy Games, Wargames"
2,115746.0,War of the Ring: Second Edition,2012.0,2,4,180,13,13725,12,4.14,22281.0,"Area Majority / Influence, Area Movement, Camp...","Thematic Games, Wargames"
3,169786.0,Scythe,2016.0,1,5,115,14,57871,14,3.41,75640.0,"Area Majority / Influence, Card Play Conflict ...",Strategy Games
4,28720.0,Brass: Lancashire,2007.0,2,4,120,14,19400,19,3.86,25429.0,"Hand Management, Income, Loans, Network and Ro...",Strategy Games


In [38]:
test_df["Domains_split"] = test_df["Domains"].str.split(", ")

test_df["Domains_split"] = test_df["Domains_split"].apply(
    lambda x: x if isinstance(x, list) else ["MISSING"]
)

domains_encoded_test = mlb_Domains.transform(test_df["Domains_split"])

domains_df_test = pd.DataFrame(
    domains_encoded_test,
    columns=[f"DOMAIN_{cls}" for cls in mlb_Domains.classes_]
)

test_df = pd.concat([test_df, domains_df_test], axis=1)
test_df = test_df.drop(columns=["Domains", "Domains_split"])

In [39]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5086 entries, 0 to 5085
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   ID                         5080 non-null   float64
 1   Name                       5086 non-null   object 
 2   Year Published             5086 non-null   float64
 3   Min Players                5086 non-null   int64  
 4   Max Players                5086 non-null   int64  
 5   Play Time                  5086 non-null   int64  
 6   Min Age                    5086 non-null   int64  
 7   Users Rated                5086 non-null   int64  
 8   BGG Rank                   5086 non-null   int64  
 9   Complexity Average         5086 non-null   float64
 10  Owned Users                5080 non-null   float64
 11  Mechanics                  4688 non-null   object 
 12  DOMAIN_Abstract Games      5086 non-null   int64  
 13  DOMAIN_Children's Games    5086 non-null   int64

In [40]:
test_df[["Mechanics"]]

,Mechanics
0,"Action Points, Cooperative Game, Hand Manageme..."
1,"Action/Event, Advantage Token, Area Majority /..."
2,"Area Majority / Influence, Area Movement, Camp..."
3,"Area Majority / Influence, Card Play Conflict ..."
4,"Hand Management, Income, Loans, Network and Ro..."
...,...
5081,"Cooperative Game, Roll / Spin and Move"
5082,Team-Based Game
5083,NaN
5084,Roll / Spin and Move


In [41]:
test_df["Mechanics_split"] = test_df["Mechanics"].str.split(", ")
test_df["Mechanics_split"] = test_df["Mechanics_split"].apply(
    lambda x: x if isinstance(x, list) else ["MISSING"]
)

test_df["Mechanics_top10"] = test_df["Mechanics_split"].apply(
    lambda lst: [m for m in lst if m in top10_mechanics]
)

mechanics_encoded_test = mlb_Mechanic.transform(test_df["Mechanics_top10"])
mechanics_df_test = pd.DataFrame(
    mechanics_encoded_test,
    columns=[f"MECHANIC_{cls}" for cls in mlb_Mechanic.classes_]
)

test_df = pd.concat([test_df, mechanics_df_test], axis=1)
test_df = test_df.drop(columns=["Mechanics", "Mechanics_split", "Mechanics_top10"])

In [42]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5086 entries, 0 to 5085
Data columns (total 30 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   ID                               5080 non-null   float64
 1   Name                             5086 non-null   object 
 2   Year Published                   5086 non-null   float64
 3   Min Players                      5086 non-null   int64  
 4   Max Players                      5086 non-null   int64  
 5   Play Time                        5086 non-null   int64  
 6   Min Age                          5086 non-null   int64  
 7   Users Rated                      5086 non-null   int64  
 8   BGG Rank                         5086 non-null   int64  
 9   Complexity Average               5086 non-null   float64
 10  Owned Users                      5080 non-null   float64
 11  DOMAIN_Abstract Games            5086 non-null   int64  
 12  DOMAIN_Children's Ga

In [43]:
test_df = test_df.drop(columns=["ID", "Name"])

In [44]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5086 entries, 0 to 5085
Data columns (total 28 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Year Published                   5086 non-null   float64
 1   Min Players                      5086 non-null   int64  
 2   Max Players                      5086 non-null   int64  
 3   Play Time                        5086 non-null   int64  
 4   Min Age                          5086 non-null   int64  
 5   Users Rated                      5086 non-null   int64  
 6   BGG Rank                         5086 non-null   int64  
 7   Complexity Average               5086 non-null   float64
 8   Owned Users                      5080 non-null   float64
 9   DOMAIN_Abstract Games            5086 non-null   int64  
 10  DOMAIN_Children's Games          5086 non-null   int64  
 11  DOMAIN_Customizable Games        5086 non-null   int64  
 12  DOMAIN_Family Games 

In [45]:
test_df.isna().sum()

Year Published                     0
Min Players                        0
Max Players                        0
Play Time                          0
Min Age                            0
Users Rated                        0
BGG Rank                           0
Complexity Average                 0
Owned Users                        6
DOMAIN_Abstract Games              0
DOMAIN_Children's Games            0
DOMAIN_Customizable Games          0
DOMAIN_Family Games                0
DOMAIN_MISSING                     0
DOMAIN_Party Games                 0
DOMAIN_Strategy Games              0
DOMAIN_Thematic Games              0
DOMAIN_Wargames                    0
MECHANIC_Card Drafting             0
MECHANIC_Dice Rolling              0
MECHANIC_Hand Management           0
MECHANIC_Hexagon Grid              0
MECHANIC_MISSING                   0
MECHANIC_Modular Board             0
MECHANIC_Set Collection            0
MECHANIC_Simulation                0
MECHANIC_Tile Placement            0
M

In [46]:
test_df["Min Players"] = test_df["Min Players"].replace(0, 1)
test_df["Max Players"] = np.where(test_df["Max Players"] == 0, test_df["Min Players"], test_df["Max Players"])

test_df["Play Time"] = test_df["Play Time"].replace(0, df["Play Time"].median())

test_df["Min Age"] = test_df["Min Age"].replace(0, df["Min Age"].median())

In [47]:
test_df.isna().sum()

Year Published                     0
Min Players                        0
Max Players                        0
Play Time                          0
Min Age                            0
Users Rated                        0
BGG Rank                           0
Complexity Average                 0
Owned Users                        6
DOMAIN_Abstract Games              0
DOMAIN_Children's Games            0
DOMAIN_Customizable Games          0
DOMAIN_Family Games                0
DOMAIN_MISSING                     0
DOMAIN_Party Games                 0
DOMAIN_Strategy Games              0
DOMAIN_Thematic Games              0
DOMAIN_Wargames                    0
MECHANIC_Card Drafting             0
MECHANIC_Dice Rolling              0
MECHANIC_Hand Management           0
MECHANIC_Hexagon Grid              0
MECHANIC_MISSING                   0
MECHANIC_Modular Board             0
MECHANIC_Set Collection            0
MECHANIC_Simulation                0
MECHANIC_Tile Placement            0
M

In [48]:
test_df["Year Published"] = test_df["Year Published"].replace(0, np.nan)
test_df["Year Published"] = test_df["Year Published"].replace(np.nan, yearMedian)

test_df["Age"] = 2025 - test_df["Year Published"]

In [49]:
test_df = test_df.drop(columns=["Year Published"])

In [50]:
test_df["Log Age"] = np.log1p(test_df["Age"])
test_df = test_df.drop(columns=["Age"])

In [51]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5086 entries, 0 to 5085
Data columns (total 28 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Min Players                      5086 non-null   int64  
 1   Max Players                      5086 non-null   int64  
 2   Play Time                        5086 non-null   int64  
 3   Min Age                          5086 non-null   int64  
 4   Users Rated                      5086 non-null   int64  
 5   BGG Rank                         5086 non-null   int64  
 6   Complexity Average               5086 non-null   float64
 7   Owned Users                      5080 non-null   float64
 8   DOMAIN_Abstract Games            5086 non-null   int64  
 9   DOMAIN_Children's Games          5086 non-null   int64  
 10  DOMAIN_Customizable Games        5086 non-null   int64  
 11  DOMAIN_Family Games              5086 non-null   int64  
 12  DOMAIN_MISSING      

In [52]:
test_df["Owned Users"] = test_df["Owned Users"].fillna(df["Owned Users"].median())

In [53]:
test_df["Max Players"] = test_df["Max Players"].clip(upper=20)
test_df["Play Time"] = test_df["Play Time"].clip(upper=600)

In [54]:
test_df.describe()

,Min Players,Max Players,Play Time,Min Age,Users Rated,BGG Rank,Complexity Average,Owned Users,DOMAIN_Abstract Games,DOMAIN_Children's Games,...,MECHANIC_Dice Rolling,MECHANIC_Hand Management,MECHANIC_Hexagon Grid,MECHANIC_MISSING,MECHANIC_Modular Board,MECHANIC_Set Collection,MECHANIC_Simulation,MECHANIC_Tile Placement,MECHANIC_Variable Player Powers,Log Age
count,5086.000000,5086.000000,5086.000000,5086.000000,5086.000000,5086.000000,5086.000000,5086.000000,5086.000000,5086.000000,...,5086.000000,5086.000000,5086.000000,5086.000000,5086.000000,5086.000000,5086.000000,5086.000000,5086.000000,5086.000000
mean,2.017696,4.837200,75.634094,10.201337,905.289029,10132.581007,1.991559,1515.774479,0.053873,0.043649,...,0.274282,0.203303,0.112466,0.078254,0.076288,0.139009,0.094967,0.079041,0.122690,2.800197
std,0.666432,2.866934,94.227616,2.716780,4113.427066,5900.401636,0.849247,5984.983288,0.225790,0.204334,...,0.446196,0.402496,0.315969,0.268597,0.265484,0.345990,0.293198,0.269828,0.328113,0.664568
min,1.000000,1.000000,1.000000,2.000000,30.000000,2.000000,0.000000,10.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.609438
25%,2.000000,4.000000,30.000000,8.000000,56.000000,5052.250000,1.330000,148.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.302585
50%,2.000000,4.000000,45.000000,10.000000,122.000000,10194.000000,1.920000,312.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.708050
75%,2.000000,6.000000,90.000000,12.000000,403.750000,15216.750000,2.560000,879.500000,0.000000,0.000000,...,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.218876
max,8.000000,20.000000,600.000000,21.000000,101853.000000,20341.000000,4.910000,154531.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,8.522380


In [55]:
test_df_scaled = mmsc.transform(test_df)
test_df_scaled = pd.DataFrame(test_df_scaled, columns=test_df.columns)

In [56]:
predictions = model.predict(test_df_scaled)

submission_df = pd.DataFrame({
    "index": range(len(predictions)),
    "Rating Average": predictions
})

submission_df.to_csv("sample_submition.csv", index=False)

In [57]:
test_df_scaled.head()

,Min Players,Max Players,Play Time,Min Age,Users Rated,BGG Rank,Complexity Average,Owned Users,DOMAIN_Abstract Games,DOMAIN_Children's Games,...,MECHANIC_Dice Rolling,MECHANIC_Hand Management,MECHANIC_Hexagon Grid,MECHANIC_MISSING,MECHANIC_Modular Board,MECHANIC_Set Collection,MECHANIC_Simulation,MECHANIC_Tile Placement,MECHANIC_Variable Player Powers,Log Age
0,0.111111,0.157895,0.098497,0.500000,0.407236,0.000049,0.568,0.420405,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.139899
1,0.111111,0.052632,0.298831,0.500000,0.399123,0.000442,0.718,0.361975,0.0,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.229324
2,0.111111,0.157895,0.298831,0.500000,0.134023,0.000541,0.828,0.143460,0.0,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.173251
3,0.000000,0.210526,0.190317,0.541667,0.566048,0.000639,0.682,0.487020,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.126718
4,0.111111,0.157895,0.198664,0.541667,0.189560,0.000885,0.772,0.163728,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.215483


In [58]:
submission_df.head()

,index,Rating Average
0,0,6.963610
1,1,7.950993
2,2,8.001931
3,3,8.206958
4,4,7.617422
